# Project Guide: BTC Range Markets vs Deribit Options

This notebook is a concise guide to the project. It is meant for a reader who wants to understand the research question, the pipeline, and the main result without reading the full codebase first.

The core idea is straightforward:

> Use the BTC options smile on Deribit to estimate fair probabilities for hourly BTC range outcomes, then compare those probabilities with executable prices on Kalshi's binary range markets.

The important result is not that the project finds easy arbitrage. The more useful result is that it shows where the apparent edge disappears:

> The model produces better-calibrated probabilities than Kalshi mid-prices, but the edge mostly vanishes once bid/ask spreads, fees, timing, and discrete option replication are included.

## 1. Research Question

Kalshi lists hourly BTC range contracts:

- Each contract pays `1 USD` if BTC settles inside a specific price range.
- Each contract pays `0 USD` otherwise.
- Example: a contract may pay if BTC settles between `80,300` and `80,399.99`.

Deribit lists BTC vanilla options:

- Calls and puts across discrete strikes and expiries.
- The option smile implies a risk-neutral distribution for future BTC prices.

The project asks:

> Are any Kalshi BTC range contracts meaningfully mispriced relative to the distribution implied by Deribit options, after realistic execution constraints?

## 2. Pipeline Overview

```text
60-second REST snapshots
        |
        v
Kalshi hourly BTC range bins      Deribit BTC option chain
        |                                  |
        |                                  v
        |                         clean quotes + infer forward
        |                                  |
        |                                  v
        |                           fit SVI option smile
        |                                  |
        +---------------> bin-level probabilities <--------------+
                                  |
                                  v
                    compare Kalshi bid/ask vs Deribit fair value
                                  |
                                  v
                         offline backtest vs settled outcomes
                                  |
                                  v
              calibration, PnL, liquidity, fees, discrete replication
```

Key files:

| Component | File |
|---|---|
| Snapshot collection | `src/snapshotter.py` |
| Main analysis pipeline | `src/runner/analyze.py` |
| SVI volatility model | `src/model/svi.py` |
| Intraday range probabilities | `src/model/intraday_q.py` |
| Discrete option replication | `src/model/replication_discrete.py` |
| Offline backtest | `src/runner/backtest.py` |
| Interactive dashboard | `src/ui/streamlit_app.py` |

## 3. Loading the Backtest Data

This notebook reads the artifacts generated by the repository.

If `data/reports/backtest.csv` does not exist yet, run this from the repository root:

```bash
bash backtest.sh
```

The notebook does not call live Deribit or Kalshi APIs. It uses local snapshots and backtest outputs so the analysis is reproducible from the saved dataset.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Allow the notebook to run both from the repository root and from notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

BACKTEST = ROOT / "data" / "reports" / "backtest.csv"
FINDINGS = ROOT / "data" / "reports" / "findings.png"

print("Repository root:", ROOT)
print("Backtest CSV exists:", BACKTEST.exists())
print("Findings image exists:", FINDINGS.exists())

In [ ]:
if not BACKTEST.exists():
    raise FileNotFoundError(
        f"Missing {BACKTEST}. Run `bash backtest.sh` from the repository root first."
    )

df = pd.read_csv(BACKTEST)
df["snap_ts"] = pd.to_datetime(df["snap_ts"], utc=True)

print(f"Bin-snapshot rows: {len(df):,}")
print(f"Snapshots: {df['snap_ts'].nunique():,}")
print(f"Kalshi events: {df['event_ticker'].nunique():,}")
print(f"Unique range contracts: {df['ticker'].nunique():,}")

df.head()

## 4. What One Backtest Row Represents

Each row is one Kalshi range contract at one snapshot timestamp, joined with its final settled outcome.

Important columns:

| Column | Meaning |
|---|---|
| `yes_bid`, `yes_ask` | Approximate executable prices on Kalshi |
| `yes_mid` | Kalshi midpoint; useful for diagnostics but not necessarily executable |
| `q_deribit` | Theoretical probability from the Deribit-implied distribution |
| `q_buy_exec`, `q_sell_exec` | Bid/ask-aware fair value proxies from the option smile |
| `q_buy_repl_disc`, `q_sell_repl_disc` | Fair value from discrete Deribit vertical spreads using real strikes |
| `outcome` | Final result: `1` if the range settled YES, `0` otherwise |

The key distinction is this:

> A midpoint may look mispriced, but a trade is only meaningful if the edge survives executable bid/ask prices, fees, and a hedge that can actually be built in Deribit.

## 5. Calibration: Does the Probability Model Predict Outcomes?

First, compare the probabilistic quality of the Deribit-implied model with Kalshi mid-prices.

- Lower Brier score is better.
- Lower log loss is better.
- `q_deribit` is compared against `yes_mid` because both are probability-like quantities.

In [ ]:
def brier(q, y):
    q = np.clip(np.asarray(q, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return np.mean((q - y) ** 2)


def log_loss(q, y):
    q = np.clip(np.asarray(q, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return -np.mean(y * np.log(q) + (1 - y) * np.log(1 - q))

metrics = pd.DataFrame([
    {
        "model": "Deribit q_deribit",
        "brier": brier(df["q_deribit"], df["outcome"]),
        "log_loss": log_loss(df["q_deribit"], df["outcome"]),
    },
    {
        "model": "Kalshi yes_mid",
        "brier": brier(df["yes_mid"], df["outcome"]),
        "log_loss": log_loss(df["yes_mid"], df["outcome"]),
    },
])

metrics

In [ ]:
def calibration_table(data, col, n_bins=10):
    d = data[[col, "outcome"]].dropna().copy()
    d["bucket"] = pd.cut(d[col], bins=np.linspace(0, 1, n_bins + 1), include_lowest=True)
    return (
        d.groupby("bucket", observed=True)
         .agg(n=("outcome", "size"), mean_pred=(col, "mean"), realized=("outcome", "mean"))
         .reset_index()
    )

cal_q = calibration_table(df, "q_deribit")
cal_k = calibration_table(df, "yes_mid")

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Perfect calibration")
ax.scatter(cal_q["mean_pred"], cal_q["realized"], s=np.sqrt(cal_q["n"]) * 10, label="Deribit q_deribit")
ax.scatter(cal_k["mean_pred"], cal_k["realized"], s=np.sqrt(cal_k["n"]) * 10, label="Kalshi yes_mid")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Realized frequency")
ax.set_title("Calibration: predicted probability vs realized outcome")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

### Interpretation

In this dataset, the Deribit-implied model is better calibrated than Kalshi mid-prices.

That does not imply a directly tradable edge. A better probability estimate can still fail as a trading signal if:

- Kalshi spreads are wide.
- The relevant Deribit hedge is not available at a tight price.
- Deribit strikes are spaced much wider than Kalshi's 100-dollar bins.
- Fees consume the signal.
- The signal appears too late or disappears before execution.

## 6. Microstructure: Can the Apparent Edge Be Traded?

Before calculating PnL, check whether the market is actually tradable.

Two variables matter immediately:

1. `yes_spread = yes_ask - yes_bid`: the cost of crossing the Kalshi spread.
2. `yes_bid_size` / `yes_ask_size`: displayed depth at the executable prices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["yes_spread"].dropna(), bins=60, range=(0, 0.5), color="#4C78A8")
axes[0].axvline(0.03, color="crimson", linestyle="--", label="3c")
axes[0].axvline(0.10, color="orange", linestyle="--", label="10c")
axes[0].set_title("Kalshi YES spread")
axes[0].set_xlabel("yes_ask - yes_bid")
axes[0].set_ylabel("Number of bin-snapshots")
axes[0].legend()
axes[0].grid(True, alpha=0.25)

axes[1].hist(df["yes_bid_size"].clip(0, 500).dropna(), bins=50, color="#59A14F")
axes[1].axvline(50, color="crimson", linestyle="--", label="50 contracts")
axes[1].set_title("Kalshi bid depth")
axes[1].set_xlabel("yes_bid_size, clipped at 500")
axes[1].set_ylabel("Number of bin-snapshots")
axes[1].legend()
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

### Interpretation

Many listed bins exist on screen but are poor trading candidates:

- The spread is too wide.
- Bid or ask depth is small or zero.
- A midpoint-based mispricing can be noise rather than executable edge.

The project therefore separates three levels of fair value:

1. **Theoretical fair value**: `q_deribit`.
2. **Bid/ask-aware fair value**: `q_buy_exec`, `q_sell_exec`.
3. **Replicable fair value using real strikes**: `q_buy_repl_disc`, `q_sell_repl_disc`.

## 7. Realistic PnL: The Main Result

The most conservative check in this notebook is:

> Sell YES on Kalshi only when `yes_bid > q_buy_repl_disc + threshold`.

This means Kalshi is paying more to sell the binary contract than the estimated cost of hedging it with a real, discrete Deribit vertical spread.

Assumptions used below:

- `threshold = 1%`
- `fee_deribit = 0.015 USD` per contract
- Kalshi fee formula from the project code
- Maximum one trade per Kalshi event, so repeated snapshots of the same event are not counted as independent opportunities

In [ ]:
from src.model.fees import kalshi_fee


def pnl_sell_yes_repl(data, threshold=0.01, fee_deribit=0.015, event_pick="first"):
    d = data.dropna(subset=["yes_bid", "yes_ask", "q_buy_repl_disc", "outcome"]).copy()
    d["sell_edge"] = d["yes_bid"] - d["q_buy_repl_disc"]
    d["sell_score"] = d["sell_edge"] - 0.5 * (d["yes_ask"] - d["yes_bid"]).fillna(0)
    trades = d[d["sell_edge"] > threshold].copy()

    if event_pick == "best":
        trades = trades.sort_values("sell_score", ascending=False).drop_duplicates("event_ticker")
    elif event_pick == "first":
        trades = trades.sort_values("snap_ts").drop_duplicates("event_ticker")
    elif event_pick == "random":
        trades = trades.sample(frac=1, random_state=42).drop_duplicates("event_ticker")
    else:
        raise ValueError("event_pick must be best, first, or random")

    trades = trades.sort_values("snap_ts")
    trades["fee_total"] = trades["yes_bid"].apply(kalshi_fee) + fee_deribit
    trades["pnl_gross"] = trades["yes_bid"] - trades["outcome"]
    trades["pnl"] = trades["pnl_gross"] - trades["fee_total"]
    trades["cum_pnl"] = trades["pnl"].cumsum()

    return trades

rows = []
trade_sets = {}
for mode in ["best", "first", "random"]:
    trades = pnl_sell_yes_repl(df, event_pick=mode)
    trade_sets[mode] = trades
    rows.append({
        "aggregation_mode": mode,
        "n_trades": len(trades),
        "pnl_net_usd": trades["pnl"].sum(),
        "avg_pnl_usd": trades["pnl"].mean() if len(trades) else np.nan,
        "win_rate": (trades["outcome"] == 0).mean() if len(trades) else np.nan,
    })

pnl_summary = pd.DataFrame(rows)
pnl_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for mode, trades in trade_sets.items():
    if not trades.empty:
        ax.plot(trades["snap_ts"], trades["cum_pnl"], marker="o", label=mode)

ax.axhline(0, color="black", linewidth=1)
ax.set_title("Cumulative net PnL by per-event aggregation mode")
ax.set_xlabel("Time")
ax.set_ylabel("Cumulative PnL, USD")
ax.grid(True, alpha=0.3)
ax.legend(title="Mode")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Interpretation

- `best` is optimistic because it chooses the best snapshot within each event after the fact.
- `first` is more realistic because it trades the first qualifying opportunity.
- `random` is a control that shows how sensitive the result is to timing.

This is the central conclusion:

> The theoretical edge is not robust once execution realism is imposed.

That is not a weak result. It is the main value of the project: the code does not stop at a pricing model; it tests whether the apparent mispricing survives market frictions.

## 8. Static Summary Figure

The repository also includes a static figure summarizing the main findings:

![Backtest findings](../data/reports/findings.png)

## 9. How to Run the Project

From the repository root:

```bash
# 1. Create the environment
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

# 2. Collect snapshots every 60 seconds
bash start.sh

# 3. Stop collection
bash stop.sh

# 4. Run the offline backtest
bash backtest.sh

# 5. Launch the interactive analytics app
bash analytics.sh
```

For a quick smoke test:

```bash
bash backtest.sh --limit 5
```

## 10. Further Research Extensions

The repository is ready to present as a research portfolio project. The remaining work is less about polish and more about extending the research toward live trading quality:

1. **Collect a larger dataset** because 20-30 hourly events are useful for demonstration but not enough for statistical significance.
2. **Use WebSocket order books for any paper/live extension**, since 60-second REST snapshots are too slow for realistic execution.
3. **Model the BRTI vs Deribit index basis** rather than assuming the two settlement references are identical.
4. **Quantify the residual tracking error** between Kalshi's rectangular binary payoff and Deribit's wider triangular vertical-spread payoff.
5. **Turn bootstrap SVI uncertainty into a trade filter** so trades are rejected when fair value is too model-dependent.
6. **Add paper-trading execution logs** before any live system: timestamps, displayed depth, simulated fills, rejected signals, and post-trade diagnostics.

The current repository already tells a strong research story: it builds a fair value model, then stress-tests whether the signal is actually tradable.